In [2]:
# training script for ingredient detection model.

# import libraries
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kritikseth/fruit-and-vegetable-image-recognition")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fruit-and-vegetable-image-recognition' dataset.
Path to dataset files: /kaggle/input/fruit-and-vegetable-image-recognition


In [19]:
os.chdir(path)
print("Current working directory:", os.getcwd())

Current working directory: /kaggle/input/fruit-and-vegetable-image-recognition


In [20]:
!ls

test  train  validation


In [22]:
# Paths

# DATA_DIR = "dataset"
DATA_DIR = path

MODEL_PATH = "models/ingredient_model_2.h5"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 30


In [ ]:
# Datagen preparation
train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,

    shear_range=0.15,
    zoom_range=0.25,

    brightness_range=(0.6, 1.4),

    channel_shift_range=40,

    horizontal_flip=True,
    vertical_flip=True,

    fill_mode='nearest'
)


# Add validation and test datagen
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)


train_gen = train_datagen.flow_from_directory(
    DATA_DIR + "/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
)

val_gen = train_datagen.flow_from_directory(
    DATA_DIR + "/validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    )
    
test_gen = test_datagen.flow_from_directory(
    DATA_DIR + "/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 3115 images belonging to 36 classes.
Found 351 images belonging to 36 classes.
Found 359 images belonging to 36 classes.


In [24]:
# Model compilation

base_model = MobileNetV2(include_top=False, input_shape=IMG_SIZE + (3,), weights="imagenet")
base_model.trainable = False  # freeze for fast training

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [25]:
# add earlystoping callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint(MODEL_PATH, save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,596 (9.26 MB)

 Trainable params: 168,612 (658.64 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

None


In [28]:
SAVE_DIR = "./saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = f"{SAVE_DIR}/ingredient_model_phase1.h5"

model.save(MODEL_PATH)
print("Saved to", MODEL_PATH)


OSError: [Errno 30] Read-only file system: './saved_models'

In [ ]:
!touch testfile.txt && echo "ok"


In [ ]:

model.save(MODEL_PATH)
print("Saved to", MODEL_PATH)


In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [26]:
# First phase: train top layers only
print("\n Phase 1: Training top layers only...")
history_1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

# save model
os.makedirs("models", exist_ok=True)
model.save(MODEL_PATH)
print(f"Model saved after Phase 1 at {MODEL_PATH}")



 Phase 1: Training top layers only...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
  6/195 ━━━━━━━━━━━━━━━━━━━━ 1:49 581ms/step - accuracy: 0.0495 - loss: 3.9361

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


195/195 ━━━━━━━━━━━━━━━━━━━━ 0s 762ms/step - accuracy: 0.2411 - loss: 2.9473

OSError: [Errno 30] Read-only file system: 'models'

In [ ]:
# Second phase: fine-tuning
print("\n Phase 2: Fine-tuning last layers of MobileNetV2...")

# unfreeze last few layers
fine_tune_layers = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_layers]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_layers:]:
    layer.trainable = True


# recompile with a lower learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5), 
    loss="categorical_crossentropy", 
    metrics=["accuracy"]
    )

history_2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)


# Save final model
model.save(MODEL_PATH)
print(f"Final fine-tuned model saved at {MODEL_PATH}")



In [ ]:
# Plot training history

acc = history_1.history['accuracy'] + history_2.history['accuracy']
val_acc = history_1.history['val_accuracy'] + history_2.history['val_accuracy']
loss = history_1.history['loss'] + history_2.history['loss']
val_loss = history_1.history['val_loss'] + history_2.history['val_loss']

# Accuracy plot
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.title('Model Accuracy (Training + Fine-tuning)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

# Loss plot
plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.title('Model Loss (Training + Fine-tuning)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

